# Laboratoire 3 — Canaux de bruit et décohérence

**Master/PhD — Informatique quantique**

Dans ce laboratoire, nous étudions les canaux de bruit quantiques et la décohérence. Nous utiliserons QuTiP, Qiskit et Stim pour simuler différents modèles de bruit.

## 1. Rappel théorique

### Opérateurs de Kraus

Un canal quantique $\mathcal{E}$ est une application linéaire complètement positive et trace-préservante (CPTP) qui agit sur un état $\rho$ via la représentation de Kraus :

$$
\mathcal{E}(\rho) = \sum_k K_k \rho K_k^\dagger, \quad \sum_k K_k^\dagger K_k = I
$$

### Équation de Lindblad

$$
\frac{d\rho}{dt} = -\frac{i}{\hbar}[H, \rho] + \sum_k \gamma_k \left( L_k \rho L_k^\dagger - \frac{1}{2}\{L_k^\dagger L_k, \rho\} \right)
$$

### Temps $T_1$ et $T_2$

- **$T_1$** : temps de relaxation longitudinale (perte d'énergie), associé à l'opérateur de Lindblad $L = \sigma_-$
- **$T_2$** : temps de déphasage transverse (perte de cohérence), associé à $L = \sigma_z$

On a la relation $1/T_2 = 1/(2T_1) + 1/T_\varphi$ où $T_\varphi$ est le temps de déphasage pur.

In [ ]:
import numpy as np
import qutip as qt
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import stim
import matplotlib.pyplot as plt

print('Bibliothèques chargées avec succès.')

## 2. Canaux avec QuTiP — Opérateurs de Kraus

### Canal dépolarisant

Le canal dépolarisant $\mathcal{D}_p$ remplace l'état par l'état maximalement mélangé avec probabilité $p$ :

$$
\mathcal{D}_p(\rho) = (1-p)\rho + p\frac{I}{2}
$$

Opérateurs de Kraus (pour un qubit) :
$$
K_0 = \sqrt{1-\frac{3p}{4}}\,I, \quad K_1 = \sqrt{\frac{p}{4}}\,X, \quad
K_2 = \sqrt{\frac{p}{4}}\,Y, \quad K_3 = \sqrt{\frac{p}{4}}\,Z
$$

In [ ]:
def canal_depolarisant_kraus(p):
    """Retourne les opérateurs de Kraus du canal dépolarisant."""
    K0 = np.sqrt(1 - 3*p/4) * qt.qeye(2)
    K1 = np.sqrt(p/4) * qt.sigmax()
    K2 = np.sqrt(p/4) * qt.sigmay()
    K3 = np.sqrt(p/4) * qt.sigmaz()
    return [K0, K1, K2, K3]

p = 0.3
K = canal_depolarisant_kraus(p)
canal = qt.to_super(K)
psi0 = qt.basis(2, 0)  # état |0⟩
rho0 = qt.ket2dm(psi0)
rho_final = qt.super_to_operator(canal).ptrace([0]) if False else qt.ket2dm(psi0)
rho_final = sum(Kk * rho0 * Kk.dag() for Kk in K)
print('État initial :')
print(rho0)
print('\nÉtat après canal dépolarisant (p=0.3) :')
print(rho_final)

### Canaux bit-flip et phase-flip

**Bit-flip** (erreur $X$) :
$$
K_0 = \sqrt{1-p}\,I, \quad K_1 = \sqrt{p}\,X
$$

**Phase-flip** (erreur $Z$) :
$$
K_0 = \sqrt{1-p}\,I, \quad K_1 = \sqrt{p}\,Z
$$

In [ ]:
def canal_bitflip_kraus(p):
    K0 = np.sqrt(1-p) * qt.qeye(2)
    K1 = np.sqrt(p) * qt.sigmax()
    return [K0, K1]

def canal_phaseflip_kraus(p):
    K0 = np.sqrt(1-p) * qt.qeye(2)
    K1 = np.sqrt(p) * qt.sigmaz()
    return [K0, K1]

def appliquer_canal(rho, K):
    return sum(Kk * rho * Kk.dag() for Kk in K)

psi_plus = (qt.basis(2, 0) + qt.basis(2, 1)).unit()
rho_plus = qt.ket2dm(psi_plus)
p = 0.2

rho_bf = appliquer_canal(rho_plus, canal_bitflip_kraus(p))
rho_pf = appliquer_canal(rho_plus, canal_phaseflip_kraus(p))

print('État |+⟩⟨+| initial :')
print(rho_plus)
print('\nAprès bit-flip (p=0.2) :')
print(rho_bf)
print('\nAprès phase-flip (p=0.2) :')
print(rho_pf)

## 3. Équation maîtresse avec `mesolve` — $T_1$ et $T_2$

Nous résolvons l'équation de Lindblad pour un qubit avec relaxation $T_1$ et déphasage $T_2$.

Hamiltonien : $H = \frac{\hbar\omega}{2} \sigma_z$

Opérateurs de Lindblad :
- $L_1 = \sqrt{1/T_1}\,\sigma_-$ (relaxation)
- $L_2 = \sqrt{1/T_\varphi}\,\sigma_z$ (déphasage pur)

In [ ]:
omega = 1.0
T1 = 10.0
T2 = 8.0
Tphi = 1.0 / (1.0/T2 - 0.5/T1)

H = 0.5 * omega * qt.sigmaz()
c_ops = [
    np.sqrt(1/T1) * qt.destroy(2),
    np.sqrt(1/Tphi) * qt.sigmaz()
]

psi0 = qt.basis(2, 1)
tlist = np.linspace(0, 50, 200)
result = qt.mesolve(H, psi0 * psi0.dag(), tlist, c_ops=c_ops, e_ops=[qt.sigmax(), qt.sigmay(), qt.sigmaz(), qt.qeye(2)])

plt.figure(figsize=(10, 6))
plt.plot(tlist, result.expect[0], label=r'$\langle X \rangle$')
plt.plot(tlist, result.expect[1], label=r'$\langle Y \rangle$')
plt.plot(tlist, result.expect[2], label=r'$\langle Z \rangle$')
plt.plot(tlist, result.expect[3], label=r'$\langle I \rangle$')
plt.xlabel('Temps')
plt.ylabel('Valeur d\'espérance')
plt.legend()
plt.title(f'Évolution de Lindblad — T₁={T1}, T₂={T2}')
plt.grid(True)
plt.show()

In [ ]:
pop_ex = result.expect[3] - result.expect[2]
plt.figure(figsize=(10, 5))
plt.semilogy(tlist, pop_ex, label='Population |1⟩')
plt.semilogy(tlist, np.exp(-tlist/T1), 'k--', label=r'$\exp(-t/T_1)$')
plt.xlabel('Temps')
plt.ylabel('Population |1⟩ (échelle log)')
plt.legend()
plt.title(f'Relaxation T₁ = {T1}')
plt.grid(True)
plt.show()

print(f'T₁ = {T1}, T₂ = {T2}, T_φ = {Tphi:.3f}')

## 4. Qiskit — NoiseModel avec canaux de bruit

Création d'un `NoiseModel` Qiskit avec canal dépolarisant sur les portes à un qubit.

In [ ]:
from qiskit_aer.noise import NoiseModel, depolarizing_error, pauli_error

noise_model = NoiseModel()
p_depol = 0.05
p_bitflip = 0.02

error_depol = depolarizing_error(p_depol, 1)
error_bitflip = pauli_error([('X', p_bitflip), ('I', 1 - p_bitflip)])

noise_model.add_all_qubit_quantum_error(error_depol, ['h', 'x', 'y', 'z', 't'])
noise_model.add_all_qubit_quantum_error(error_bitflip, ['sx'])

print(noise_model)

In [ ]:
qr = QuantumRegister(1, 'q')
cr = ClassicalRegister(1, 'c')
qc = QuantumCircuit(qr, cr)
qc.h(0)
qc.t(0)
qc.sx(0)
qc.measure(0, 0)

sim_ideal = AerSimulator()
sim_noisy = AerSimulator(noise_model=noise_model)

qc_t = transpile(qc, sim_ideal)
result_ideal = sim_ideal.run(qc_t, shots=4096).result()
counts_ideal = result_ideal.get_counts()

qc_t = transpile(qc, sim_noisy)
result_noisy = sim_noisy.run(qc_t, shots=4096).result()
counts_noisy = result_noisy.get_counts()

print('Résultats idéaux :', counts_ideal)
print('Résultats bruités :', counts_noisy)

## 5. Stim — Modèle de bruit Pauli

Stim utilise des opérations avec des `DEPOLARIZE1`, `X_ERROR`, `Z_ERROR` pour simuler du bruit Pauli au niveau des circuits.

In [ ]:
c = stim.Circuit()
c.append('H', 0)
c.append('DEPOLARIZE1', 0, 0.01)
c.append('CNOT', [0, 1])
c.append('DEPOLARIZE2', [0, 1], 0.02)
c.append('M', [0, 1])

sampler = c.compile_sampler()
samples = sampler.sample(100)

print('Échantillons (100 shots) :')
print(samples[:10, :], '...')
print(f'\nFréquence des mesures à 1 : {np.mean(samples[:, 0]):.3f}')

## 6. Comparaison circuit idéal vs bruité

Nous comparons l'effet du bruit sur la préparation d'un état $|+\rangle$ et la mesure en base $X$.

In [ ]:
def comparer_bruit(taux_depol):
    qr = QuantumRegister(1, 'q')
    cr = ClassicalRegister(1, 'c')
    qc = QuantumCircuit(qr, cr)
    qc.h(0)
    qc.sx(0)
    qc.h(0)
    qc.measure(0, 0)

    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(taux_depol, 1), ['h', 'sx'])

    sim = AerSimulator(noise_model=nm)
    qc_t = transpile(qc, sim)
    result = sim.run(qc_t, shots=8192).result()
    counts = result.get_counts()
    p1 = counts.get('1', 0) / 8192
    return p1

taux = np.linspace(0, 0.5, 10)
probas_1 = [comparer_bruit(t) for t in taux]

plt.figure(figsize=(8, 5))
plt.plot(taux, probas_1, 'o-', label='P(mesure = 1)')
plt.axhline(0.5, color='k', linestyle='--', label='Valeur idéale (0.5)')
plt.xlabel('Taux de dépolarisation')
plt.ylabel('Probabilité de mesurer 1')
plt.legend()
plt.grid(True)
plt.title('Effet du bruit dépolarisant sur une mesure')
plt.show()

## 7. Questions et exercices

### Questions théoriques

1. **Opérateurs de Kraus.** Vérifiez que les opérateurs de Kraus du canal dépolarisant satisfont $\sum_k K_k^\dagger K_k = I$. Que se passe-t-il lorsque $p = 1$ ?

2. **Canal bit-flip vs phase-flip.** Appliquez le canal bit-flip et phase-flip à l'état $|0\rangle$. Quelle est la différence sur la matrice densité résultante ?

3. **Relaxation T₁.** Que vaut $\langle Z \rangle$ à l'équilibre ($t \to \infty$) dans la simulation `mesolve` ? Justifiez.

4. **Déphasage pur.** Modifiez $T_\varphi$ pour que $T_2 = 5$ avec $T_1 = 10$. Simulation l'évolution de $\langle X \rangle$ et commentez.

### Exercices pratiques

5. **Canal d'amplitude damping.** Implémentez le canal d'amplitude damping avec QuTiP (opérateurs de Kraus $K_0 = |0\rangle\langle 0| + \sqrt{1-\gamma}|1\rangle\langle 1|$, $K_1 = \sqrt{\gamma}|0\rangle\langle 1|$) et appliquez-le à $|1\rangle$.

6. **Fidélité.** Calculez la fidélité $F(\rho, \sigma) = \operatorname{Tr}\sqrt{\rho^{1/2}\sigma\rho^{1/2}}$ entre l'état idéal et l'état bruité pour différents taux de bruit avec QuTiP (`qt.fidelity`).

7. **NoiseModel Qiskit personnalisé.** Créez un `NoiseModel` avec $T_1 = 50\,\mu s$ et $T_2 = 30\,\mu s$ en utilisant `thermal_relaxation_error` de Qiskit. Appliquez-le à un circuit de 3 qubits et comparez les résultats avec le cas idéal.

8. **Stim — détection d'erreurs.** Utilisez Stim pour générer des échantillons d'un circuit avec des erreurs Pauli et implémentez un détecteur de syndrome (stabilisateurs) pour un code de répétition à 3 qubits.